In [0]:
-- Medallion Arch
    -- Bronze
    -- Silver
    -- Gold

In [0]:
-- Input Data - many sources - many file formats (csv/json/xml)
-- Convert into Delta (Parquet + delta_log)

In [0]:
-- Bronze Layer

In [0]:
-- Raw Data -- Landing Zone 

In [0]:
%fs ls /Volumes/dea_databricks_catalog/default/dea_volume/yellowtaxi_raw/csv/

path,name,size,modificationTime
dbfs:/Volumes/dea_databricks_catalog/default/dea_volume/yellowtaxi_raw/csv/yellow_tripdata_2019_01.csv,yellow_tripdata_2019_01.csv,1341900,1777273593000


In [0]:
-- Ways-- 
-- 1 - Read raw data in a DF 

In [0]:
%python
input_path = '/Volumes/dea_databricks_catalog/default/dea_volume/yellowtaxi_raw/csv/'

In [0]:
%python
yt_raw_DF = spark.read.option("Header","True").option("InferSchema","True").option("delim", ",").format("csv").load("/Volumes/dea_databricks_catalog/default/dea_volume/yellowtaxi_raw/csv/")

In [0]:
show catalogs

catalog
dea_cat_2
dea_databricks_catalog
dlt
medallion_arch
python_cat
samples
sn_tr_ws
system
tr_catalog


In [0]:
use catalog dea_databricks_catalog

In [0]:
show databases

databaseName
default
information_schema


In [0]:
create database nyctaxi_med_db

In [0]:
use nyctaxi_med_db

In [0]:
show tables

database,tableName,isTemporary


In [0]:
%python
yt_raw_DF.write.saveAsTable("yt_bronze_delta_1")

In [0]:
-- 2 - To read raw data in a view

In [0]:
create or replace temp view yt_raw_view
using csv
options
(
  path '/Volumes/dea_databricks_catalog/default/dea_volume/yellowtaxi_raw/csv/',
  header 'true',
  inferSchema 'true'
)

In [0]:
show tables

database,tableName,isTemporary
nyctaxi_med_db,yt_bronze_delta_1,false
,yt_raw_view,true


In [0]:
%python
yt_raw_DF.count()

15000

In [0]:
select count(*) from yt_raw_view

count(*)
79999


In [0]:
create or replace table yt_bronze_delta_2
as
select * from yt_raw_view; 

num_affected_rows,num_inserted_rows


In [0]:
show tables

database,tableName,isTemporary
nyctaxi_med_db,yt_bronze_delta_1,false
nyctaxi_med_db,yt_bronze_delta_2,false
,yt_raw_view,true


In [0]:
create table yt_raw_csv
using csv
options
(
  path 'dbfs:/Volumes/dea_databricks_catalog/default/dea_volume/yellowtaxi_raw/csv/',
  header 'true',
  inferSchema 'true'
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5912662621149453>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "create table yt_raw_csv\nusing csv\noptions\n(\n  path 'dbfs:/Volumes/dea_databricks_catalog/default/dea_volume/yellowtaxi_raw/csv/',\n  header 'true',\n  inferSchema 'true'\n)\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntim

In [0]:
select count(*) from yt_bronze_delta_1

count(*)
15000


In [0]:
select count(*) from yt_bronze_delta_1

count(*)
15000


In [0]:
select count(*) from yt_raw_view

count(*)
15000


In [0]:
show tables

database,tableName,isTemporary
nyctaxi_med_db,yt_bronze_delta_1,false
nyctaxi_med_db,yt_bronze_delta_2,false
,yt_raw_view,true


In [0]:
-- refresh table dea_databricks_catalog.nyctaxi_med_db.yt_bronze_delta_1

In [0]:
select count(*) from yt_bronze_delta_2

count(*)
30000


In [0]:
select count(*) from yt_bronze_delta_2 version as of 0 

count(*)
15000


In [0]:
insert into yt_bronze_delta_2 select * from yt_raw_view 

num_affected_rows,num_inserted_rows
30000,30000


In [0]:
describe history yt_bronze_delta_2

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-04-28T04:46:58.000Z,3245098896349135,snagpure888@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> true)",null,List(203710676150460),0e5cade5-bfbb-4a28-a8d9-deaf2c093b0d,0428-035351-apmmy5m3-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 457472, numOutputRows -> 30000)",null,Databricks-Runtime/18.1.x-photon-scala2.13
1,2026-04-28T04:44:03.000Z,3245098896349135,snagpure888@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(203710676150460),3dc982d0-b266-4eb8-b630-ce5b4292bd61,0428-035351-apmmy5m3-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 215602, numDeletionVectorsRemoved -> 0, numOutputRows -> 30000, numOutputBytes -> 455024)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-04-28T04:37:31.000Z,3245098896349135,snagpure888@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(203710676150460),34526b83-d8fd-49f2-9eff-c99085e02a67,0428-035351-apmmy5m3-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 15000, numOutputBytes -> 215602)",null,Databricks-Runtime/18.1.x-photon-scala2.13


In [0]:
-- Silver Layer

In [0]:
desc yt_bronze_delta_2

col_name,data_type,comment
VendorID,int,null
tpep_pickup_datetime,timestamp,null
tpep_dropoff_datetime,timestamp,null
passenger_count,int,null
trip_distance,double,null
RatecodeID,int,null
store_and_fwd_flag,string,null
PULocationID,int,null
DOLocationID,int,null
payment_type,int,null


In [0]:
select count(*) from yt_bronze_delta_2

count(*)
60000


In [0]:
create or replace table yt_silver_delta
as
select * from yt_bronze_delta_2 version as of 1
where 
passenger_count > 0
and
trip_distance > 0
and
fare_amount > 0
and
total_amount > 0
and
tpep_dropoff_datetime > tpep_pickup_datetime
and
PULocationID is not null
and
DOLocationID is not Null


num_affected_rows,num_inserted_rows


In [0]:
select count(*) from yt_silver_delta

count(*)
29317


In [0]:
desc history yt_silver_delta

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-04-28T04:57:18.000Z,3245098896349135,snagpure888@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(203710676150460),5598df0d-7330-412c-a198-a82a6ed1c516,0428-035351-apmmy5m3-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 627369, numDeletionVectorsRemoved -> 0, numOutputRows -> 29317, numOutputBytes -> 444539)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-04-28T04:56:05.000Z,3245098896349135,snagpure888@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(203710676150460),2ebb4390-ccb5-4dfb-a04f-f5f9d5bc682a,0428-035351-apmmy5m3-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 58634, numOutputBytes -> 627369)",null,Databricks-Runtime/18.1.x-photon-scala2.13


In [0]:
-- Gold Layer

In [0]:
ALTER TABLE std SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'false',
  'delta.autoOptimize.autoCompact' = 'false'
);

In [0]:
SHOW TBLPROPERTIES yt_silver_delta ('delta.autoOptimize.optimizeWrite')

key,value
delta.autoOptimize.optimizeWrite,Table dea_databricks_catalog.nyctaxi_med_db.yt_silver_delta does not have property: delta.autoOptimize.optimizeWrite


In [0]:
create or replace table yt_gold_total_revenue_per_vendor_delta
as
select VendorID, sum(Total_amount) as Total_Revenue from yt_silver_delta
group by VendorID

num_affected_rows,num_inserted_rows


In [0]:
select * from yt_gold_total_revenue_per_vendor_delta

VendorID,Total_Revenue
1,164718.59999999567
2,303574.50999994593
4,4331.000000000012


In [0]:
create or replace table yt_gold_stats_delta
as
select 
VendorID, 
sum(Total_amount) as Total_Revenue,
avg(Total_amount) as Avg_Revenue,
min(Total_amount) as Min_Revenue,
max(Total_amount) as Max_Revenue,
sum(passenger_count) as Total_Passengers,
avg(passenger_count) as Avg_Passengers,
min(passenger_count) as Min_Passengers,
max(passenger_count) as Max_Passengers,
sum(trip_distance) as Total_Distance,
avg(trip_distance) as Avg_Distance,
min(trip_distance) as Min_Distance,
max(trip_distance) as Max_Distance,
sum(fare_amount) as total_fare,
avg(fare_amount) as avg_fare,
min(fare_amount) as min_fare,
max(fare_amount) as max_fare
from yt_silver_delta
group by VendorID

num_affected_rows,num_inserted_rows


In [0]:
select * from yt_gold_stats_delta

VendorID,Total_Revenue,Avg_Revenue,Min_Revenue,Max_Revenue,Total_Passengers,Avg_Passengers,Min_Passengers,Max_Passengers,Total_Distance,Avg_Distance,Min_Distance,Max_Distance,total_fare,avg_fare,min_fare,max_fare
1,164718.59999999567,15.791256830600677,0.31,300.3,14478,1.3879781420765027,1,6,31522.399999999867,3.0219921388169753,0.1,52.6,131047.52,12.56327485380117,0.01,300.0
2,303574.50999994593,16.323843092969078,3.8,453.44,34572,1.8590095176641395,1,6,59751.05999999993,3.2129407969027226,0.01,128.73,241336.24999999997,12.977160294671181,2.5,450.0
4,4331.000000000012,14.986159169550215,3.8,70.27,298,1.0311418685121108,1,3,790.9399999999995,2.7368166089965382,0.06,20.5,3395.0,11.747404844290658,2.5,52.0


In [0]:
-- Insert , Update, Delete, Merge (UpSert) 

In [0]:
-- Upsert - Merge

Syntax

MERGE INTO target_table AS t
USING source_table AS s
ON t.id = s.id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
desc yt_raw_view

col_name,data_type,comment
VendorID,int,null
tpep_pickup_datetime,timestamp,null
tpep_dropoff_datetime,timestamp,null
passenger_count,int,null
trip_distance,double,null
RatecodeID,int,null
store_and_fwd_flag,string,null
PULocationID,int,null
DOLocationID,int,null
payment_type,int,null


In [0]:
select count(*) from yt_bronze_delta_2

count(*)
60000


In [0]:
-- Use Merge on Bronze tables only

In [0]:
MERGE INTO yt_bronze_delta_2 AS t
USING yt_raw_view AS s
ON t.tpep_pickup_datetime = s.tpep_pickup_datetime
WHEN NOT MATCHED THEN
  INSERT *


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
49999,0,0,49999


In [0]:
select count(*) from yt_bronze_delta_2

count(*)
109999


In [0]:
select count(*) from yt_silver_delta

count(*)
29317


In [0]:
select count(*) from yt_bronze_delta_2 
where 
passenger_count > 0
and
trip_distance > 0
and
fare_amount > 0
and
total_amount > 0
and
tpep_dropoff_datetime > tpep_pickup_datetime
and
PULocationID is not null
and
DOLocationID is not Null


count(*)
107215


In [0]:

insert into yt_silver_delta
select * from yt_bronze_delta_2
where 
passenger_count > 0
and
trip_distance > 0
and
fare_amount > 0
and
total_amount > 0
and
tpep_dropoff_datetime > tpep_pickup_datetime
and
PULocationID is not null
and
DOLocationID is not Null


num_affected_rows,num_inserted_rows
107215,107215


In [0]:
-- End of the Notebook